<a href="https://colab.research.google.com/github/lankipolo123/roadfixqc/blob/main/retrain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# ROADFIX YOLO11 - RESUME TRAINING & AUTO-DOWNLOAD
# =============================================================================

# 1. Fix SymPy compatibility issue
!pip uninstall sympy -y -q
!pip install sympy>=1.12 -q

# 2. Mount Drive & Install dependencies
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!pip install ultralytics roboflow opencv-python -q

# 3. Imports
import os
from pathlib import Path
from ultralytics import YOLO
from roboflow import Roboflow
import torch
from google.colab import files

# 4. GPU check
if not torch.cuda.is_available():
    raise SystemExit("❌ Enable GPU first! Runtime → Change runtime type → T4 GPU")

print("✅ GPU Available:", torch.cuda.get_device_name(0))

# 5. Download Dataset from Roboflow
rf = Roboflow(api_key="SzttdelfmuWaCwAz2N5u")
project = rf.workspace("dequillaprojects").project("roadfix-model-jycpr")
dataset = project.version(2).download("yolov11", location="/content/RoadFixDataset")

dataset_path = Path("/content/RoadFixDataset")

# 6. Create data.yaml
yaml_path = dataset_path / "data.yaml"
yaml_content = f"""
train: {dataset_path}/train/images
val:   {dataset_path}/valid/images

nc: 11
names: ["Compromised_Pole", "Fallen-Barrier", "Fallen-Cone", "Pothole",
        "Road-Cracks", "Road_Barrier", "Sewage-Manhole", "Stable",
        "Tires", "Tires_with_rim", "Traffic_Cones"]
"""
yaml_path.write_text(yaml_content)

# 7. Setup training directories
drive_path = Path("/content/drive/MyDrive/roadfix_training")
os.makedirs(drive_path, exist_ok=True)

# 8. Check for existing checkpoint to resume
best_ckpt = drive_path / "roadfix_v2/weights/best.pt"
last_ckpt = drive_path / "roadfix_v2/weights/last.pt"

if last_ckpt.exists():
    resume_ckpt = str(last_ckpt)
    resume_mode = True
    print(f"🔄 RESUMING from: {last_ckpt}")
elif best_ckpt.exists():
    resume_ckpt = str(best_ckpt)
    resume_mode = True
    print(f"🔄 RESUMING from: {best_ckpt}")
else:
    resume_ckpt = "yolo11n.pt"
    resume_mode = False
    print("🆕 Starting NEW training with yolo11n.pt")

# 9. Train/Resume Model
model = YOLO(resume_ckpt)
results = model.train(
    data=str(yaml_path),
    epochs=200,
    imgsz=1024,
    batch=32,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    device=0,
    workers=8,
    amp=True,
    save_period=25,
    project=str(drive_path),
    name='roadfix_v2',
    exist_ok=True,
    resume=resume_mode
)

# 10. Auto-download trained model
print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)

final_best = drive_path / "roadfix_v2/weights/best.pt"
final_last = drive_path / "roadfix_v2/weights/last.pt"

if final_best.exists():
    print(f"\n📥 Downloading best.pt...")
    files.download(str(final_best))

if final_last.exists():
    print(f"📥 Downloading last.pt...")
    files.download(str(final_last))

print("\n✅ Models saved in Google Drive:")
print(f"   {drive_path}/roadfix_v2/weights/")
print("\n✅ Models downloaded to your computer!")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 131.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ GPU Available: NVIDIA A100-SXM4-80GB
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/RoadFixDataset in yolov11:: 100%|██████████| 135942/135942 [00:19<00:00, 6999.71it/s] 


🔄 RESUMING from: /content/drive/MyDrive/roadfix_training/roadfix_v2/weights/last.pt
Ultralytics 8.3.227 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/RoadFixDataset/data.yaml, degrees=0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=/content/drive/MyDrive/roadfix_training/roadfix_v2/weights/last.pt, momentum=0.937, mosaic=1.0, multi_sc

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Downloading last.pt...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Models saved in Google Drive:
   /content/drive/MyDrive/roadfix_training/roadfix_v2/weights/

✅ Models downloaded to your computer!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')